# 이벤트 전처리: 필수 이벤트만 남기기

윈도우로 나누기 **전에** 이벤트 단위로 정리한다.

```
build_labels.py → [이 노트북] → windows.py → features.py
event_labels.csv → events_preprocessed.csv → windows_*.csv
```

| 단계 | 내용 |
|---|---|
| 1 | 서비스 이벤트 제외: AWS 서비스가 대신 호출한 이벤트 (EKS, Auto Scaling 등) |
| 2 | 반복 압축: 같은 주체가 같은 호출을 연달아 반복하면 1개로 합치고 반복 횟수를 남김 |
| 3 | 누출 제거: 라벨을 알려주는 필드(리전, userAgent, 이름, IP)를 일반화 |

규칙은 **라벨을 보지 않고** 이벤트 필드만으로 정한다. 실제 탐지 때도 같은 규칙을 쓸 수 있어야 하기 때문이다.
(예외: 반복 압축에서 공격·정상 이벤트가 하나로 합쳐지지 않도록 `attack_id`를 키에 넣는다.)

In [1]:
import ipaddress
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IN = ROOT / "data" / "interim" / "event_labels.csv"
OUT = ROOT / "data" / "interim" / "events_preprocessed.csv"

CFG = {
    "drop_service_events": True,
    "compress_repeats": True,
    "max_repeat_gap_sec": 300,   # 이 간격보다 멀리 떨어진 반복은 따로 둔다
}

In [2]:
events = pd.read_csv(IN)
events["eventTime"] = pd.to_datetime(events["eventTime"], utc=True)
events = events.sort_values(["session", "eventTime", "seq"], kind="stable").reset_index(drop=True)


def summary(df, name):
    n_att = int(df["is_attack"].sum())
    print(f"{name:12s} 이벤트 {len(df):6d}  공격 {n_att:5d}  정상 {len(df) - n_att:6d}  "
          f"기법 {df['technique'].nunique():2d}  세션 {df['session'].nunique()}")


summary(events, "원본")

원본           이벤트  21998  공격  8058  정상  13940  기법 29  세션 77


## 1. 서비스 이벤트 제외

AWS 서비스가 사용자를 대신해 호출한 이벤트는 사람이나 공격자의 행위가 아니다.

- `userType == "AWSService"` 또는 ARN에 `AWSServiceRole`
- 호출 출처(`sourceIPAddress`)와 `userAgent`가 **둘 다** `*.amazonaws.com` (예: `eks.amazonaws.com`)
  - 콘솔(`console.*`, `signin.*`)은 사람이 쓰는 것이므로 제외하지 않는다.

In [3]:
def is_service_event(df):
    ip = df["sourceIPAddress"].fillna("")
    ua = df["userAgent"].fillna("")
    arn = df["userArn"].fillna("")
    via_service = ip.str.endswith(".amazonaws.com") & ua.str.endswith(".amazonaws.com") & ~ip.str.contains("console|signin")
    return df["userType"].eq("AWSService") | arn.str.contains("AWSServiceRole") | via_service


svc = is_service_event(events)
print("서비스 이벤트:", svc.sum())
print(events.loc[svc, "sourceIPAddress"].value_counts().head(8).to_string())
print("\n제외되는 공격 이벤트:")
print(events.loc[svc & events["is_attack"], ["session", "eventName", "technique"]].to_string())

if CFG["drop_service_events"]:
    events = events[~svc].reset_index(drop=True)
summary(events, "서비스 제외")

서비스 이벤트: 11928
sourceIPAddress
eks.amazonaws.com                10711
autoscaling.amazonaws.com         1040
eks-nodegroup.amazonaws.com         69
config.amazonaws.com                61
trustedadvisor.amazonaws.com        17
access-analyzer.amazonaws.com       15
lambda.amazonaws.com                 6
ec2.amazonaws.com                    3

제외되는 공격 이벤트:
                                     session    eventName  technique
19858  splunk:aws_invoke_model_access_denied  InvokeModel  T1087.004
서비스 제외       이벤트  10070  공격  8057  정상   2013  기법 28  세션 76


## 2. 반복 압축

같은 세션에서 **같은 주체**가 **같은 호출**(서비스, 이름, 에러코드)을 연달아 하면 첫 이벤트 1개만 남긴다.

- 다른 주체의 이벤트가 사이에 끼어 있어도 주체별 흐름 안에서 연속이면 합친다.
- 직전 같은 호출과 `max_repeat_gap_sec`보다 멀리 떨어지면 새 묶음으로 본다.
- 압축 전 정보는 `repeat_count`, `last_eventTime`, `duration_sec`로 남긴다. 윈도우 특징(`features.py`)에서 쓴다.

In [4]:
def compress_repeats(df, max_gap_sec):
    df = df.copy()
    actor = df["userArn"].fillna(df["userType"].fillna("unknown"))
    key = [df["session"], actor]
    call = (df["eventSource"].fillna("") + "|" + df["eventName"].fillna("") + "|"
            + df["errorCode"].fillna("") + "|" + df["attack_id"].fillna(""))

    prev_call = call.groupby(key).shift()
    gap = df["eventTime"].groupby(key).diff().dt.total_seconds()
    new_run = (call != prev_call) | (gap > max_gap_sec)
    run_id = new_run.groupby(key).cumsum()

    df["_run"] = df["session"] + "|" + actor + "|" + run_id.astype(str)
    agg = df.groupby("_run").agg(repeat_count=("_run", "size"), last_eventTime=("eventTime", "max"))
    first = df.drop_duplicates("_run").set_index("_run").join(agg)
    first["duration_sec"] = (first["last_eventTime"] - first["eventTime"]).dt.total_seconds()
    return first.reset_index(drop=True).sort_values(["session", "eventTime", "seq"], kind="stable").reset_index(drop=True)


before = events
if CFG["compress_repeats"]:
    events = compress_repeats(events, CFG["max_repeat_gap_sec"])
else:
    events = events.assign(repeat_count=1, last_eventTime=events["eventTime"], duration_sec=0.0)
summary(events, "반복 압축")

print("\n가장 많이 압축된 호출:")
print(events.nlargest(10, "repeat_count")[["session", "eventName", "is_attack", "repeat_count", "duration_sec"]].to_string())

반복 압축        이벤트   4591  공격  3109  정상   1482  기법 28  세션 76

가장 많이 압축된 호출:
                                               session     eventName  is_attack  repeat_count  duration_sec
56    splunk:aws_cloudwatch_logs_high_volume_retrieval  GetLogEvents       True           168           8.0
2504       splunk:aws_iam_excessive_list_command_usage   ListObjects       True           152       33899.0
2681       splunk:aws_iam_excessive_list_command_usage   ListObjects       True           111       24600.0
2552       splunk:aws_iam_excessive_list_command_usage   ListObjects       True           109       24298.0
1425                splunk:aws_exfil_high_no_getobject     GetObject       True           100           0.0
3310                                splunk:aws_kms_key       Decrypt      False            92        4423.0
20    splunk:aws_cloudwatch_logs_high_volume_retrieval  GetLogEvents       True            86           1.0
2599       splunk:aws_iam_excessive_list_command_usage   ListO

기법별 이벤트 수 변화. 이벤트가 많던 탐색 계열 기법이 크게 줄어 기법 간 불균형이 완화된다.

In [5]:
per_tech = pd.DataFrame({
    "원본": before[before["is_attack"]]["technique"].value_counts(),
    "압축 후": events[events["is_attack"]]["technique"].value_counts(),
}).fillna(0).astype(int).sort_values("원본", ascending=False)
per_tech

,원본,압축 후
technique,,
T1580,5360,1694
T1526,1071,939
T1530,759,57
T1098,245,179
T1078,126,99
T1651,85,8
T1552,60,2
T1552.GEN,53,6
T1555.006,49,8


## 3. 누출 제거

라벨을 정할 때는 썼지만 모델 입력에 들어가면 정답을 알려주는 필드들이다.

| 필드 | 문제 | 처리 |
|---|---|---|
| `awsRegion` | Stratus 리전은 `ap-central-2r`처럼 **가짜 리전**이라 리전만 보고 공격을 맞힐 수 있다 | 제거하고, 세션의 주 리전과 다른지만 `region_changed`로 남긴다 (T1535 신호) |
| `userAgent` | `stratus-red-team_<uuid>` | 도구 종류(`ua_kind`)로 일반화. Stratus는 Go SDK로 동작하므로 `sdk-go` |
| `userArn` | `rhino_escalate`, `stratus-red-team-...`, `cloudsploit` 같은 시나리오용 이름 | 세션 안에서만 의미 있는 가명(`IAMUser-1`)으로 바꾼다 |
| `sourceIPAddress` | 익명화 방식이 소스마다 다름 (Stratus는 `211.9.016.253`처럼 형식이 깨진 IP) | 종류(`ip_kind`)와 세션 내 가명(`ip-1`)으로 바꾼다 |
| `eventID`, `source_file`, `scenario` | 식별자 | 모델 입력에서 뺀다 (추적용으로 `event_uid`만 유지) |

In [6]:
UA_RULES = [
    ("stratus-red-team", "sdk-go"),
    ("terraform", "terraform"), ("cloudformation", "cloudformation"),
    ("console", "console"), ("signin", "console"), ("mozilla", "browser"),
    ("aws-cli", "aws-cli"), ("boto", "sdk-python"),
    ("aws-sdk-go", "sdk-go"), ("aws-sdk-java", "sdk-java"), ("aws-sdk-nodejs", "sdk-js"),
    ("aws-sdk-dotnet", "sdk-dotnet"), ("aws-internal", "aws-internal"),
    (".amazonaws.com", "aws-service"),
]


def ua_kind(ua):
    ua = str(ua).lower() if isinstance(ua, str) else ""
    return next((kind for pat, kind in UA_RULES if pat in ua), "other" if ua else "none")


def ip_kind(ip):
    if not isinstance(ip, str) or not ip:
        return "none"
    if ip.endswith(".amazonaws.com") or ip == "AWS Internal":
        return "aws-service"
    try:
        return "private" if ipaddress.ip_address(ip).is_private else "public"
    except ValueError:
        return "public" if re.fullmatch(r"[\d.]+|[0-9a-fA-F:]+", ip) else "other"


def pseudonym(df, col, prefix):
    """세션 안에서 처음 나온 순서대로 번호를 붙인다."""
    p = prefix if isinstance(prefix, pd.Series) else pd.Series(prefix, index=df.index)
    order = df.groupby("session")[col].transform(lambda s: pd.Series(pd.factorize(s)[0] + 1, index=s.index))
    return (p + "-" + order.astype(str)).where(df[col].notna())


ev = events.copy()
main_region = ev.groupby("session")["awsRegion"].transform(lambda s: s.mode().iat[0] if s.notna().any() else None)
ev["region_changed"] = ev["awsRegion"].ne(main_region) & ev["awsRegion"].notna()
ev["ua_kind"] = ev["userAgent"].map(ua_kind)
ev["ip_kind"] = ev["sourceIPAddress"].map(ip_kind)
ev["actor"] = pseudonym(ev, "userArn", ev["userType"].fillna("unknown"))
ev["ip"] = pseudonym(ev, "sourceIPAddress", "ip")

print(pd.crosstab(ev["ua_kind"], ev["source"]).to_string())

source      splunk  stratus
ua_kind                    
aws-cli        294        0
browser         40        0
console        162        0
none             1        0
other            7        0
sdk-dotnet     267        0
sdk-go          17       76
sdk-java       878        0
sdk-js        2250        0
sdk-python     402        0
terraform      197        0


남은 문자열 필드에 누출 문자열이 없는지 확인한다.

In [7]:
MODEL_COLS = [
    "event_uid", "session", "seq", "eventTime", "last_eventTime", "duration_sec", "repeat_count",
    "eventSource", "eventName", "readOnly", "errorCode",
    "userType", "actor", "ua_kind", "ip_kind", "ip", "region_changed",
]
LABEL_COLS = ["source", "is_attack", "attack_id", "technique", "tactics", "label_level", "benign_type"]

out = ev[MODEL_COLS + LABEL_COLS]
leak = re.compile(r"stratus|red-team|rhino|cloudsploit|cloudmapper|atomic", re.I)
model_text = out[[c for c in MODEL_COLS if c not in ("event_uid", "session")]].astype(str)
hits = model_text.apply(lambda s: s.str.contains(leak)).sum()
print("누출 문자열이 남은 필드:", hits[hits > 0].to_dict() or "없음")

누출 문자열이 남은 필드: 없음


In [8]:
out.to_csv(OUT, index=False)
summary(out, "최종")
print(f"saved {OUT.relative_to(ROOT)}")

최종           이벤트   4591  공격  3109  정상   1482  기법 28  세션 76
saved data/interim/events_preprocessed.csv
